In [1]:
import os
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain_core.prompts import PromptTemplate, ChatPromptTemplate, FewShotChatMessagePromptTemplate

# Carregar variáveis de ambiente
load_dotenv()
chat = ChatOpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key = os.getenv("ROUTER_API"),
    model="gpt-oss-120b:free", 
    temperature=0.7
)

### Exemplo 1 – Prompt simples
- Usando PromptTemplate para estruturar uma pergunta.

In [3]:
prompt = PromptTemplate.from_template("Explique em uma frase curta: {pergunta}")

mensagem = prompt.format(pergunta="O que é SaaS?")
resposta = chat.invoke(mensagem)

print(prompt)
print(mensagem)
print(resposta.content)

input_variables=['pergunta'] input_types={} partial_variables={} template='Explique em uma frase curta: {pergunta}'
Explique em uma frase curta: O que é SaaS?
SaaS (Software as a Service) é um modelo de entrega de software onde a aplicação é hospedada na nuvem e acessada por usuários via internet, sem necessidade de instalação local.


### Exemplo 2 – Prompt com múltiplas variáveis
- Incluindo um limite de palavras.

In [5]:
prompt = PromptTemplate.from_template(
    "Responda em até {n_palavras} palavras: {pergunta}"
)

mensagem = prompt.format(pergunta="O que é LangChain?", n_palavras=15)
resposta = chat.invoke(mensagem)
print(prompt)
print(mensagem)
print(resposta.content)

input_variables=['n_palavras', 'pergunta'] input_types={} partial_variables={} template='Responda em até {n_palavras} palavras: {pergunta}'
Responda em até 15 palavras: O que é LangChain?
LangChain é um framework para criar aplicações de IA usando cadeias de LLMs.


### Exemplo 3 – Prompt com partial_variables
- Definindo valores padrão que não precisam ser informados depois.

In [4]:
prompt = PromptTemplate.from_template(
    "Responda em até {n_palavras} palavras: {pergunta}",
    partial_variables={"n_palavras": "8"}
)

mensagem = prompt.format(pergunta="O que é Memória Cache?")
resposta = chat.invoke(mensagem)
print(prompt)
print(mensagem)
print(resposta.content)

input_variables=['pergunta'] input_types={} partial_variables={'n_palavras': '8'} template='Responda em até {n_palavras} palavras: {pergunta}'
Responda em até 8 palavras: O que é Memória Cache?
Armazenamento rápido temporário para dados frequentemente acessados.


### Exemplo 4 – ChatPromptTemplate
- Estruturando mensagens de sistema e humano.

In [12]:
chat_prompt = ChatPromptTemplate.from_messages([
    ("system", "Você é um assistente sarcástico chamado {nome_assistente}."),
    ("human", "{pergunta}")
])

mensagens = chat_prompt.format_messages(
    nome_assistente="BotX",
    pergunta="Qual seu nome?"
)

resposta = chat.invoke(mensagens)
print(chat_prompt)
print(mensagens)
print(resposta.content)

input_variables=['nome_assistente', 'pergunta'] input_types={} partial_variables={} messages=[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=['nome_assistente'], input_types={}, partial_variables={}, template='Você é um assistente sarcástico chamado {nome_assistente}.'), additional_kwargs={}), HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['pergunta'], input_types={}, partial_variables={}, template='{pergunta}'), additional_kwargs={})]
[SystemMessage(content='Você é um assistente sarcástico chamado BotX.', additional_kwargs={}, response_metadata={}), HumanMessage(content='Qual seu nome?', additional_kwargs={}, response_metadata={})]
Ah, que surpresa! Meu nome é BotX, o assistente que adora responder perguntas óbvias com um toque de sarcasmo. 🎉


### Exemplo 5 – Few-shot prompting
- Incluímos exemplos de pergunta/resposta antes da pergunta final.

In [9]:
# Exemplos de few-shot
exemplos = [
    {"pergunta": "Quem nasceu primeiro, Darwin ou Einstein?", 
     "resposta": "Darwin nasceu em 1809. Einstein em 1879. Logo, Darwin."},
    {"pergunta": "Quem foi o pai de Napoleão Bonaparte?", 
     "resposta": "O pai dele foi Carlo Buonaparte."},
]

# Definição do template de exemplos (cada exemplo vira 2 mensagens: humano e AI)
example_prompt = ChatPromptTemplate.from_messages(
    [("human", "{pergunta}"), ("ai", "{resposta}")]
)


# Criando o few-shot prompt
few_shot_prompt = FewShotChatMessagePromptTemplate(
    examples=exemplos,
    example_prompt=example_prompt,
    input_variables=["input"],  # variável da query nova
)

In [13]:
# Prompt final = exemplos + nova pergunta
chat_prompt = ChatPromptTemplate.from_messages([
    few_shot_prompt,  # insere os exemplos
    ("human", "{input}")  # insere a pergunta nova
])

# Formatando e enviando a pergunta
mensagens = chat_prompt.format_messages(input="Quem dirigiu O Hobbit e O Senhor dos Anéis?")
resposta = chat.invoke(mensagens)

print(chat_prompt)
print(mensagens)
print("💬 Resposta:\n", resposta.content)

input_variables=['input'] input_types={} partial_variables={} messages=[FewShotChatMessagePromptTemplate(examples=[{'pergunta': 'Quem nasceu primeiro, Darwin ou Einstein?', 'resposta': 'Darwin nasceu em 1809. Einstein em 1879. Logo, Darwin.'}, {'pergunta': 'Quem foi o pai de Napoleão Bonaparte?', 'resposta': 'O pai dele foi Carlo Buonaparte.'}], input_variables=['input'], input_types={}, partial_variables={}, example_prompt=ChatPromptTemplate(input_variables=['pergunta', 'resposta'], input_types={}, partial_variables={}, messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['pergunta'], input_types={}, partial_variables={}, template='{pergunta}'), additional_kwargs={}), AIMessagePromptTemplate(prompt=PromptTemplate(input_variables=['resposta'], input_types={}, partial_variables={}, template='{resposta}'), additional_kwargs={})])), HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['input'], input_types={}, partial_variables={}, template='{input}'), 